# **2. Tagueamento híbrido das justificativas de cartão**

Este notebook substitui o pipeline anterior (embeddings + grafo + Leiden + redução hierárquica) por uma
abordagem auditável que combina os **códigos disciplinares oficiais** com **regras lexicais explícitas**
sobre os textos já limpos no notebook 1. Cada evento pode receber um ou vários rótulos (multirrótulo).
Os códigos fornecem a classificação principal e o texto detalha códigos genéricos como `A1.24` e `V2.8`.

A troca é intencional: o pipeline de embeddings produzia grupos estatisticamente coerentes mas difíceis de
auditar (por que um texto caiu num cluster e não noutro?) e caros de reexecutar (modelo de embedding, UMAP,
Leiden). A taxonomia híbrida preserva a transparência, registra a origem de cada rótulo e mantém
`NAO_CLASSIFICADA` somente como indicador de auditoria, não como uma conduta nos gráficos.

## **1. Dados**

A classificação é feita por **evento**, porque 275 textos iguais aparecem associados a mais de um código
oficial. Depois, os resultados são agregados por `reason_key` para produzir a auditoria de textos únicos
sem atribuir indevidamente o código de um evento a outro.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "card_events.csv"
OUTPUT_DIR = ROOT / "data" / "processed" / "keyword_tags"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(DATA_PATH, low_memory=False)
required_columns = {"event_id", "reason_key", "reason", "official_code"}
missing_columns = sorted(required_columns.difference(events.columns))
if missing_columns:
    raise ValueError(f"Colunas ausentes em {DATA_PATH.name}: {missing_columns}")

valid_events = events.loc[events["reason_key"].notna() & events["reason"].notna()].copy()
texts = (
    valid_events.groupby("reason_key", as_index=False)
    .agg(reason=("reason", "first"), event_count=("event_id", "size"))
    .sort_values("reason_key")
    .reset_index(drop=True)
)

display(
    pd.DataFrame(
        {
            "metric": ["eventos", "eventos com texto", "textos únicos"],
            "value": [len(events), len(valid_events), len(texts)],
        }
    )
)

,metric,value
0,eventos,13757
1,eventos com texto,13757
2,textos únicos,4134


## **2. Taxonomia híbrida**

As categorias têm nomes curtos para os gráficos, famílias temáticas, códigos oficiais e padrões regex sobre
o texto normalizado (`reason`: sem acento, minúsculo e sem stopwords). Um evento recebe a união dos rótulos
obtidos pelo código e pelo texto. Assim, uma entrada que impede uma oportunidade clara pode carregar mais
de um rótulo. Casos ainda não explicados recebem `NAO_CLASSIFICADA` e vão para auditoria.

O código oficial é a fonte primária; as regras lexicais complementam o código, principalmente quando a
súmula usa os códigos genéricos `A1.24` ou `V2.8`. A taxonomia inteira vive neste notebook — não há módulo
Python externo — para que a fonte da verdade fique junto da análise que a usa.

In [2]:
UNCLASSIFIED = "NAO_CLASSIFICADA"

# ``label`` é deliberadamente curto: é o texto usado nos gráficos.
# ``family`` permite separar visualizações com muitas categorias.
TAXONOMY = {
    "ACAO_TEMERARIA": {
        "label": "Ação temerária",
        "family": "Faltas e infrações de jogo",
        "official_codes": ["A1.1", "A1.2", "A1.9", "A1.10", "A1.11", "A1.12", "A1.13"],
        "patterns": [
            r"\btemerari[ao]\b",
            r"\bimprudente\b",
            r"\b(?:calcar|calco|rasteira|carrinho|pontape|tranco|solad[ao]|tesoura)\b",
            r"\b(?:dar|deu|desferir)\b.*\bentrada\b.*\badversario\b",
            r"\b(?:entrada|golpear|atingir|chutar|carga)\b.*\bdisputa bola\b",
        ],
    },
    "FALTA_TATICA": {
        "label": "Impedir ataque promissor",
        "family": "Faltas e infrações de jogo",
        "official_codes": ["A1.3", "A1.4", "A1.5"],
        "patterns": [
            r"\b(?:ataque|contra ataque) promissor\b",
            r"\bataque prometedor\b",
            r"\bfalta tatica\b",
            r"\bmotivo tatico\b",
            r"\bimped\w*\b.*\b(?:ataque promissor|progressao|posse bola)\b",
            r"\b(?:segurar|agarrar|puxar)\b.*\bafasta lo bola\b",
        ],
    },
    "IMPEDIR_OPORTUNIDADE_GOL": {
        "label": "Impedir chance clara de gol",
        "family": "Faltas e infrações de jogo",
        "official_codes": ["V4"],
        "patterns": [
            r"\b(?:impedir|impedindo|acabar)\b.*\b(?:oportunidade|chance) clara\b.*\bgol\b",
            r"\b(?:oportunidade|chance) clara\b.*\bgol\b",
            r"\b(?:impedir|impedindo|evitando)\b.*\b(?:clara|manifesta)\b.*\b(?:oportunidade|chance)\b.*\bgol\b",
            r"\b(?:impedir|impedindo|evitando)\b.*\b(?:oportunidade|chance)\b.*\bgol\b",
            r"\bimpedir clara oportunidade gol\b",
            r"\bimpedir\b.*\bgol\b.*\b(?:mao|falta|segurar|calcar|derrubar)\b",
        ],
    },
    "JOGO_BRUSCO_GRAVE": {
        "label": "Jogo brusco grave",
        "family": "Faltas e infrações de jogo",
        "official_code_prefixes": ["V1"],
        # V2.7 é um erro de catalogação da própria CBF: seu texto oficial diz
        # "...na disputa da bola" (igual a V1.7), não "fora da disputa da
        # bola" como o resto da família V2 (conduta violenta). Semanticamente
        # é jogo brusco grave, então tratamos o código como uma exceção aqui.
        "official_codes": ["V2.7"],
        "patterns": [
            r"\bjogo brusco grave\b",
            r"\bforca excessiva\b.*\bdisputa bola\b",
            r"\bdisputa bola\b.*\bforca excessiva\b",
            r"\bentrada\b.*\b(?:sola|solada|tesoura)\b.*\b(?:forca excessiva|intensidade alta)\b",
            r"\batingir\b.*\btravas chuteira\b.*\bforca excessiva\b",
        ],
    },
    "MAO_ANTIDESPORTIVA": {
        "label": "Mão na bola",
        "family": "Faltas e infrações de jogo",
        "official_codes": ["A1.5", "A1.6", "V4"],
        "patterns": [
            r"\b(?:tocar|jogar|bloquear|interceptar)\b.*\bbola\b.*\b(?:mao|braco)\b.*\b(?:impedir|impedindo|tentando|tentar|marcar)\b",
            r"\bmao\b.*\b(?:impedir|impedindo)\b.*\b(?:ataque|gol|chute|meta)\b",
            r"\b(?:mao|braco)\b.*\bbola\b.*\b(?:impedir|impedindo)\b.*\b(?:ataque|gol|chute|meta)\b",
            r"\buso (?:intencional|deliberado) mao\b",
            r"\bbola\b.*\b(?:mao|braco)\b.*\bbloquear\b",
        ],
    },
    "USO_IRREGULAR_BRACO": {
        "label": "Uso irregular do braço",
        "family": "Faltas e infrações de jogo",
        "patterns": [
            r"\bcotovelada\b",
            r"\b(?:mao|braco|antebraco)\b.*\brosto\b",
            r"\brosto\b.*\b(?:mao|braco|antebraco)\b",
            r"\bgolpear\b.*\b(?:braco|antebraco)\b",
            r"\batingir\b.*\b(?:braco|antebraco)\b.*\b(?:rosto|nuca|cabeca|pescoco|costas)\b",
            r"\buso indevido\b.*\bbraco\w*\b",
        ],
    },
    "INFRACAO_PERSISTENTE": {
        "label": "Infração persistente",
        "family": "Faltas e infrações de jogo",
        "official_code_prefixes": ["A4"],
        "patterns": [
            r"\bpersistente(?:mente)?\b.*\b(?:infra[cç][aã]o|infracoes|falta|faltas|regras)\b",
            r"\b(?:infra[cç][aã]o|infracoes|falta|faltas|regras)\b.*\bpersistente(?:mente)?\b",
            r"\binfringir\b.*\bregras\b",
            r"\bnumero diferentes infracoes\b",
            r"\bnumero excessivo faltas\b",
            r"\brodizio faltas\b",
            r"\bpersistir\b.*\b(?:faltas|infracoes)\b",
        ],
    },
    "RECLAMAR_ARBITRAGEM": {
        "label": "Reclamação à arbitragem",
        "family": "Comportamento e disciplina",
        "official_code_prefixes": ["A2"],
        "patterns": [
            r"\b(?:reclam|desaprov|protest)\w*\b",
            r"\b(?<!sem )(?:discordar|contestar|question\w*)\b.*\b(?:decis|arbitragem|arbitro)\w*\b",
            r"\bdecis(?:ao|oes) arbitragem\b",
            r"\bgestos?\b.*\b(?:arbitragem|arbitro)\b",
            r"\b(?:bater palmas|aplaudir)\b.*\b(?:ironic|arbitro|arbitragem)\w*\b",
            r"\bbat\w* palmas\b.*\b(?:ironic|arbitro|arbitragem)?\w*\b",
            r"\b(?:direcao|dirigir)\b.*\b(?:arbitro|arbitragem|assistente)\b.*\b(?:ostensiv|agressiv|dedo riste)\w*\b",
            r"\bdirecao arbitro\b.*\b(?:ironic|desrespe)\w*\b",
            r"\bgestos? (?:roubo|roubando)\b|\bsinal conhecido como roubo\b",
        ],
    },
    "CONDUTA_VIOLENTA": {
        "label": "Conduta violenta",
        "family": "Comportamento e disciplina",
        "official_codes": ["V2", "V2.1", "V2.2", "V2.3", "V2.4", "V2.5", "V2.6"],
        "patterns": [
            r"\bconduta violenta\b",
            r"\bforca excessiva\b.*\bfora (?:disputa )?bola\b",
            r"\bfora (?:disputa )?bola\b.*\bforca excessiva\b",
            r"\bforca excessiva\b.*\bbola\b.*\bfora jogo\b",
            r"\b(?:soco|socos|socar|tapa|tapas|cabecada|morder|mordida|cuspir|cusparada|agredir|agressao)\b",
            r"\b(?:golpear|atingir|chutar|empurrar)\b.*\b(?:rosto|cabeca|pescoco|nuca)\b.*\bforca excessiva\b",
            r"\b(?:apertar|segurar|agarrar|atingir)\b.*\b(?:pescoco|nuca)\b",
            r"\b(?:peitada|atingir peito|acertando peito gandula)\b",
            r"\bempurrar\b.*\b(?:rosto|pescoco)\b.*\bfora disputa bola\b",
            r"\bpisar\b.*\bpe\b",
        ],
    },
    "LINGUAGEM_OFENSIVA": {
        "label": "Ofensa verbal ou gestual",
        "family": "Comportamento e disciplina",
        "patterns": [
            r"\b(?:ofensiv[ao]s?|ofend\w*|ofensa|injurios[ao]s?|insulto|insultos)\b",
            r"\b(?:xingar|xingamento|xingou)\b",
            r"\bgesto (?:obsceno|obceno|ofensivo)\b",
            r"\b(?:caralho|porra|merda|foda|foder|puta|palhacada|ladrao|vergonha|sacanagem|sacana|babaca|incompetente|incompetencia|vagabundo|veado|cuzao)\b",
            r"\b(?:vai|vao|toma|tomar|teu)\b.*\bcu\b|\bpau cu\b",
            r"\bvoce (?:muito )?(?:fraco|ruim|mal intencionado)\b",
            r"\bvoces? sao (?:muito )?(?:fracos?|ruins?)\b",
            r"\bnao servem nem apitar\b",
            r"\b(?:roubando|roubado|roubo)\b.*\b(?:arbitro|arbitragem|gesto|maos?|ofendido)\b",
        ],
    },
    "CONFRONTO_TUMULTO": {
        "label": "Confronto ou tumulto",
        "family": "Comportamento e disciplina",
        "official_codes": ["A8.2"],
        "patterns": [
            r"\b(?:confronto|confrontar|tumulto|confusao|briga|conflito|conflitivo|desentendimento|desentender|desintendimento)\b",
            r"\btrocar\b.*\b(?:empurr\w*|emperr\w*|empuro\w*|socos|ofensas|insultos|provoc\w*)\b",
            r"\bdiscut\w*\b.*\b(?:adversari\w*|banco reservas|acintos\w*)\b",
            r"\bempurrar\b.*\b(?:adversario|atleta)\b.*\b(?:provocando|iniciando|gerando)\b",
            r"\bempurrar\b.*\b(?:adversario|equipe adversaria)\b.*\barea tecnica\b",
            r"\bprovoc\w*\b",
            r"\bgestos?\b.*\b(?:provocador|provocadores|debochado|debochados)\b",
            r"\bdeboche\b|\bdebochar\b",
            r"\b(?:partir|partiu|partindo)\w*\b.*\b(?:cima|direcao)\b.*\badversari\w*\b",
            # Contato físico com o adversário (segurar/agarrar/puxar/empurrar,
            # com variações de conjugação e erros de digitação comuns nas
            # súmulas) só sinaliza confronto quando acontece fora da disputa
            # normal da bola (jogo/partida parada, atendimento médico) ou é
            # descrito como deliberadamente hostil ("de forma acintosa") —
            # caso contrário é só uma falta tática de jogo, não um princípio
            # de briga.
            r"\b(?:segur|agarr|pux|empurr|emburr|emnpurr|emperr|empuro)\w*\b.*\badversari\w*\b.*\b(?:parad[ao]\w*|paralisad\w*|atendimento)\b",
            r"\b(?:parad[ao]\w*|paralisad\w*|atendimento)\b.*\b(?:segur|agarr|pux|empurr|emburr|emnpurr|emperr|empuro)\w*\b.*\badversari\w*\b",
            r"\b(?:segur|agarr|pux|empurr|emburr|emnpurr|emperr|empuro)\w*\b.*\b(?:parad[ao]\w*|paralisad\w*|atendimento)\b.*\badversari\w*\b",
            r"\b(?:segur|agarr|pux|empurr|emburr|emnpurr|emperr|empuro)\w*\b.*\badversari\w*\b.*\bfora\b.*\b(?:jogo|disputa|posse)\b",
            r"\bfora\b.*\b(?:jogo|disputa|posse)\b.*\b(?:segur|agarr|pux|empurr|emburr|emnpurr|emperr|empuro)\w*\b.*\badversari\w*\b",
        ],
    },
    "SIMULACAO": {
        "label": "Simulação",
        "family": "Comportamento e disciplina",
        "official_codes": ["A1.7"],
        "patterns": [
            r"\b(?:simular|simulacao|simulando)\b",
            r"\b(?:fingir|enganar)\b.*\b(?:arbitro|falta|lesao)\b",
        ],
    },
    "CONDUTA_ANTIDESPORTIVA": {
        "label": "Conduta antidesportiva",
        "family": "Comportamento e disciplina",
        "official_codes": ["A1", "A1.14", "A1.17", "A1.24"],
        "patterns": [
            r"\banti[\s\w]*esport\w*\b",
            r"\batitude inconveniente\b",
            r"\bmarcas? nao autorizadas?\b",
            r"\badiantar\b.*\bexecucao tiro penal\b",
            r"\bsair banco reservas\b.*\b(?:ostensiv|agressiv|dedo riste)\w*\b",
            r"\bdes\w*speit\w*\b",
            r"\bfalta respeito\b",
            r"\bdespeito\b.*\bjogo\b",
            # "de forma/maneira acintosa" qualifica o contato físico como
            # deliberadamente desleal — diferente do contato bruto genérico,
            # que vira confronto (ver CONFRONTO_TUMULTO).
            r"\b(?:segur|agarr|pux|toc)\w*\b.*\b(?:forma|maneira) acintosa\w*\b",
        ],
    },
    "RETARDAR_REINICIO": {
        "label": "Retardar o jogo",
        "family": "Infrações procedimentais",
        "official_codes": ["A5"],
        "official_code_prefixes": ["A3"],
        "patterns": [
            r"\bretard\w*\b",
            r"\breinicio\b",
            r"\b(?:chutar|jogar|arremessar|lancar|carregar) bola\b.*\b(?:longe|paralisad[ao]|apito|marcacao falta)\b",
            r"\bbola longe\b.*\b(?:apito|paralisad[ao]|marcacao falta)\b",
            r"\bdemorar\b.*\b(?:reinicio|cobranca|execucao|saida|substituicao)\b",
            r"\bimpedir\b.*\b(?:reinicio|reposicao bola|cobranca)\b",
            r"\bnao respeitar\b.*\bdistancia regulamentar\b",
            r"\bdistancia regulamentar\b.*\b(?:tiro|cobranca|escanteio|lateral)\b",
            r"\bganhar tempo\b",
            r"\bforcar\b.*\batendimento\b",
        ],
    },
    "COMEMORACAO_IRREGULAR": {
        "label": "Comemoração irregular",
        "family": "Infrações procedimentais",
        "official_codes": ["A8", "A8.3", "A8.4"],
        "patterns": [
            r"\b(?:tirar|retirar) camisa\b",
            r"\b(?:sem camisa|cobrir cabeca camisa)\b",
            r"\balambrado\b",
            r"\bcomemora\w*\b.*\bgol\b",
        ],
    },
    "AREA_TECNICA": {
        "label": "Infração na área técnica",
        "family": "Infrações procedimentais",
        "patterns": [
            r"\b(?:sair|sa[ií]\w*|abandonar|invadir|fora|nao respeitar)\b.*\barea tecnica\b",
            r"\barea tecnica\b.*\b(?:sair|deixar|abandonar|invadir|limite|permanecer|reclamar|protestar|confrontar)\w*\b",
            r"\blevantar\b.*\bbanco reservas\b.*\b(?:reclamar|protestar|gesticular|dirigir)\w*\b",
            r"\bbanco reservas\b.*\b(?:descumprir|descumprindo|determinacoes arbitragem)\b",
        ],
    },
    "ENTRADA_SAIDA_CAMPO": {
        "label": "Entrada/saída sem permissão",
        "family": "Infrações procedimentais",
        "official_codes": ["A1.15", "A6", "A7"],
        "patterns": [
            r"\b(?:entrar|retornar|abandonar|invadir|sair)\b.*\bcampo(?: jogo)?\b.*\bsem (?:minha )?(?:permissao|autorizacao)\b",
            r"\bsem (?:permissao|autorizacao)\b.*\b(?:entrar|retornar|abandonar|invadir|sair)\b.*\bcampo\b",
            r"\binvadiu?\b.*\bcampo(?: jogo)?\b",
        ],
    },
}


SEMANTIC_TAGS = list(TAXONOMY)
TAGS = SEMANTIC_TAGS + [UNCLASSIFIED]
TAG_LABELS = {tag: spec["label"] for tag, spec in TAXONOMY.items()} | {
    UNCLASSIFIED: "Não classificada"
}
TAG_FAMILIES = {tag: spec["family"] for tag, spec in TAXONOMY.items()} | {
    UNCLASSIFIED: "Qualidade do tagueamento"
}


_NEGATED_RECLAMAR = re.compile(r"\bsem\s+(?:contestar|discordar|question\w*|reclam\w*|protest\w*)\b")

_COMPILED = {
    tag: [re.compile(pattern) for pattern in spec.get("patterns", [])]
    for tag, spec in TAXONOMY.items()
}


class TaggingResult(NamedTuple):
    tags: frozenset[str]
    source: str
    matched_rules: tuple[str, ...]


def _normalize_code(value: object) -> str:
    if value is None:
        return ""
    code = str(value).strip().upper()
    return "" if code in {"", "NAN", "NONE"} else code


def _code_matches(code: str, spec: dict) -> bool:
    if not code:
        return False
    if code in spec.get("official_codes", []):
        return True
    return any(
        code == prefix or code.startswith(f"{prefix}.")
        for prefix in spec.get("official_code_prefixes", [])
    )


def classify_reason(reason: object, official_code: object = None) -> TaggingResult:
    """Classifica um evento e devolve rótulos, origem e regras acionadas."""

    text = "" if reason is None else str(reason).strip().casefold()
    if text == "nan":
        text = ""
    code = _normalize_code(official_code)

    code_tags = {tag for tag, spec in TAXONOMY.items() if _code_matches(code, spec)}
    text_tags: set[str] = set()
    matched_rules: list[str] = []
    for tag, patterns in _COMPILED.items():
        for index, pattern in enumerate(patterns):
            if pattern.search(text):
                text_tags.add(tag)
                matched_rules.append(f"text:{tag}:{index + 1}")
                break

    # A1.24 significa uma conduta antidesportiva não especificada pelo sistema.
    # Se o texto revelar a conduta, preferimos o rótulo específico; caso
    # contrário, o rótulo geral continua sendo uma classificação honesta.
    if code == "A1.24" and text_tags - {"CONDUTA_ANTIDESPORTIVA"}:
        code_tags.discard("CONDUTA_ANTIDESPORTIVA")

    # A1.14 ("desrespeito ao jogo") é o código genérico de contato físico
    # não autorizado com o adversário sem uma intenção mais específica
    # descrita no texto (mesmo raciocínio do A1.24 acima). Se o texto já
    # aponta algo mais específico (confronto, ação temerária etc.), esse
    # rótulo mais específico prevalece; caso contrário, "conduta
    # antidesportiva" é a classificação honesta.
    if code == "A1.14" and text_tags - {"CONDUTA_ANTIDESPORTIVA"}:
        code_tags.discard("CONDUTA_ANTIDESPORTIVA")

    # "Ação temerária" é o rótulo mais genérico da família de faltas de
    # jogo — jogo brusco grave, mão na bola, uso irregular do braço e
    # impedir chance clara de gol são leituras mais específicas do mesmo
    # tipo de lance. Ela só é mantida junto com uma dessas categorias mais
    # específicas quando o próprio texto usa as palavras explícitas
    # "temerária"/"imprudente" (padrões 1-2); se veio só do código oficial
    # ou de um padrão genérico de entrada/carrinho (padrões 3-5), o rótulo
    # mais específico prevalece sozinho.
    SPECIFIC_OVER_TEMERARIA = {
        "JOGO_BRUSCO_GRAVE", "IMPEDIR_OPORTUNIDADE_GOL",
        "USO_IRREGULAR_BRACO", "MAO_ANTIDESPORTIVA", "FALTA_TATICA",
    }
    explicit_temeraria = {
        r for r in matched_rules
        if r.startswith("text:ACAO_TEMERARIA:") and r.rsplit(":", 1)[-1] in ("1", "2")
    }
    combined_tags = code_tags | text_tags
    if (
        "ACAO_TEMERARIA" in combined_tags
        and SPECIFIC_OVER_TEMERARIA & combined_tags
        and not explicit_temeraria
    ):
        code_tags.discard("ACAO_TEMERARIA")
        text_tags.discard("ACAO_TEMERARIA")

    if "RECLAMAR_ARBITRAGEM" in text_tags and _NEGATED_RECLAMAR.search(text):
        direct_reclaim = {r for r in matched_rules if r == "text:RECLAMAR_ARBITRAGEM:1"}
        if not direct_reclaim:
            text_tags.discard("RECLAMAR_ARBITRAGEM")

    tags = code_tags | text_tags
    if not tags:
        return TaggingResult(frozenset({UNCLASSIFIED}), "unclassified", ())

    matched_rules.extend(f"code:{code}:{tag}" for tag in sorted(code_tags))
    if code_tags and text_tags:
        source = "official_code+text"
    elif code_tags:
        source = "official_code"
    else:
        source = "text"
    return TaggingResult(frozenset(tags), source, tuple(sorted(matched_rules)))


taxonomy_summary = pd.DataFrame(
    [{"tag": tag, "label": TAG_LABELS[tag], "family": TAG_FAMILIES[tag]} for tag in SEMANTIC_TAGS]
)
print(f"{len(SEMANTIC_TAGS)} categorias + indicador {UNCLASSIFIED}")
display(taxonomy_summary)

17 categorias + indicador NAO_CLASSIFICADA


,tag,label,family
0,ACAO_TEMERARIA,Ação temerária,Faltas e infrações de jogo
1,FALTA_TATICA,Impedir ataque promissor,Faltas e infrações de jogo
2,IMPEDIR_OPORTUNIDADE_GOL,Impedir chance clara de gol,Faltas e infrações de jogo
3,JOGO_BRUSCO_GRAVE,Jogo brusco grave,Faltas e infrações de jogo
4,MAO_ANTIDESPORTIVA,Mão na bola,Faltas e infrações de jogo
5,USO_IRREGULAR_BRACO,Uso irregular do braço,Faltas e infrações de jogo
6,INFRACAO_PERSISTENTE,Infração persistente,Faltas e infrações de jogo
7,RECLAMAR_ARBITRAGEM,Reclamação à arbitragem,Comportamento e disciplina
8,CONDUTA_VIOLENTA,Conduta violenta,Comportamento e disciplina
9,LINGUAGEM_OFENSIVA,Ofensa verbal ou gestual,Comportamento e disciplina


### Verificação de sanidade da taxonomia

Casos de referência que travam regressões silenciosas nas regras acima: precedência do código oficial,
normalização (sem acento/stopword), multirrótulo em DOGSO com mão antidesportiva, prioridade do texto sobre
`A1.24` genérico e um código genérico sem pista textual permanecendo `NAO_CLASSIFICADA` para auditoria.

In [3]:
def _tags(reason, code=None):
    return classify_reason(reason, code).tags


sanity_cases = [
    ("código oficial é a classificação primária",
     _tags("texto sem palavras chave", "A1.3"), {"in": "FALTA_TATICA"}),
    ("texto normalizado (sem acento/stopword) ainda casa a regra",
     _tags("chutar bola longe depois arbitro ter paralisado jogo"), {"in": "RETARDAR_REINICIO"}),
    ("DOGSO com mão antidesportiva é multirrótulo",
     _tags("impedir oportunidade clara gol uso intencional mao bola", "V4"),
     {"in_all": ["IMPEDIR_OPORTUNIDADE_GOL", "MAO_ANTIDESPORTIVA"]}),
    ("confronto detectado e A1.24 genérico cede ao rótulo específico",
     _tags("trocar empurroes adversario bola fora jogo", "A1.24"),
     {"in": "CONFRONTO_TUMULTO", "not_in": "CONDUTA_ANTIDESPORTIVA"}),
    ("A1.24 sem pista textual é um fallback válido",
     _tags("ocorrencia sem detalhe", "A1.24"), {"eq": frozenset({"CONDUTA_ANTIDESPORTIVA"})}),
    ("código genérico desconhecido permanece auditável",
     _tags("ocorrencia sem detalhe", "V2.8"), {"eq": frozenset({UNCLASSIFIED})}),
    ("linguagem ofensiva flexionada é reconhecida",
     _tags("voces sao muito ruins nao servem nem apitar serie c", "V2.8"),
     {"in": "LINGUAGEM_OFENSIVA"}),
    ("distância regulamentar é agrupada como retardar o jogo",
     _tags("nao respeitar distancia regulamentar tiro livre", "A5"),
     {"in": "RETARDAR_REINICIO", "not_in": "DISTANCIA_REGULAMENTAR"}),
]

for description, tags, expectation in sanity_cases:
    if "eq" in expectation:
        assert tags == expectation["eq"], f"{description}: {tags}"
    if "in" in expectation:
        assert expectation["in"] in tags, f"{description}: {tags}"
    if "in_all" in expectation:
        assert all(tag in tags for tag in expectation["in_all"]), f"{description}: {tags}"
    if "not_in" in expectation:
        assert expectation["not_in"] not in tags, f"{description}: {tags}"

print(f"{len(sanity_cases)} casos de sanidade OK")

8 casos de sanidade OK


## **3. Aplicação das regras**

Cada evento combina os rótulos encontrados no código oficial e no texto normalizado. A origem (`tag_source`)
e as regras acionadas (`matched_rules`) são preservadas para auditoria. Em seguida, os rótulos dos eventos são
agregados por texto único para medir cobertura e coocorrência; nenhuma etapa usa modelo estatístico.

In [4]:
classification = valid_events.apply(
    lambda row: classify_reason(row["reason"], row["official_code"]), axis=1
)
valid_events["tags"] = classification.map(lambda result: result.tags)
valid_events["tag_source"] = classification.map(lambda result: result.source)
valid_events["matched_rules"] = classification.map(lambda result: ";".join(result.matched_rules))
for tag in TAGS:
    valid_events[tag] = valid_events["tags"].map(lambda tag_set, t=tag: t in tag_set)

classified_columns = ["event_id", "tag_source", "matched_rules"] + TAGS
events_tagged = events.merge(
    valid_events[classified_columns], on="event_id", how="left", validate="one_to_one"
)
for tag in TAGS:
    events_tagged[tag] = events_tagged[tag].fillna(tag == UNCLASSIFIED).astype(bool)

text_tag_matrix = events_tagged.groupby("reason_key")[SEMANTIC_TAGS].any()
texts = texts.drop(columns=[column for column in TAGS if column in texts], errors="ignore")
texts = texts.merge(text_tag_matrix, on="reason_key", how="left", validate="one_to_one")
for tag in SEMANTIC_TAGS:
    texts[tag] = texts[tag].fillna(False).astype(bool)
texts[UNCLASSIFIED] = ~texts[SEMANTIC_TAGS].any(axis=1)
texts["n_tags"] = texts[SEMANTIC_TAGS].sum(axis=1)

text_provenance = (
    valid_events.groupby("reason_key")
    .agg(
        tag_source=("tag_source", lambda values: ";".join(sorted(set(values)))),
        matched_rules=("matched_rules", lambda values: ";".join(sorted({rule for value in values for rule in value.split(";") if rule}))),
    )
)
texts = texts.merge(text_provenance, on="reason_key", how="left", validate="one_to_one")
texts.to_csv(OUTPUT_DIR / "unique_text_tags.csv", index=False)
events_tagged.to_csv(OUTPUT_DIR / "card_events_tagged.csv", index=False)

coverage = pd.Series(
    {
        "textos_com_categoria": (texts["n_tags"] > 0).mean(),
        "eventos_com_categoria": (~events_tagged[UNCLASSIFIED]).mean(),
        "textos_nao_classificados": texts[UNCLASSIFIED].mean(),
        "eventos_nao_classificados": events_tagged[UNCLASSIFIED].mean(),
        "cardinalidade_media_rotulados": events_tagged.loc[~events_tagged[UNCLASSIFIED], SEMANTIC_TAGS].sum(axis=1).mean(),
    },
    name="value",
)
display(coverage.round(3).to_frame())
display(texts.loc[texts["n_tags"] > 0, "n_tags"].value_counts().sort_index().rename("textos").to_frame())

,value
textos_com_categoria,1.000
eventos_com_categoria,1.000
textos_nao_classificados,0.000
eventos_nao_classificados,0.000
cardinalidade_media_rotulados,1.172


,textos
n_tags,
1,2751
2,1190
3,171
4,18
5,4


## **4. Frequência por categoria e casos não classificados**

Frequência de cada categoria em textos únicos e eventos. Em seguida, todos os textos ainda não classificados
são ordenados pela frequência — a fila de auditoria que indica categorias ou regras ainda ausentes.

In [5]:
tag_frequency = pd.DataFrame(
    {
        "tag": TAGS,
        "label": [TAG_LABELS[tag] for tag in TAGS],
        "family": [TAG_FAMILIES[tag] for tag in TAGS],
        "n_textos": [int(texts[tag].sum()) for tag in TAGS],
        "n_eventos": [int(events_tagged[tag].sum()) for tag in TAGS],
    }
).sort_values("n_eventos", ascending=False).reset_index(drop=True)
tag_frequency.to_csv(OUTPUT_DIR / "tag_frequency.csv", index=False)
display(tag_frequency)

unclassified_texts = texts.loc[texts[UNCLASSIFIED]].sort_values("event_count", ascending=False)
unclassified_texts[["reason", "event_count", "tag_source"]].to_csv(
    OUTPUT_DIR / "unclassified_texts.csv", index=False
)
display(unclassified_texts[["reason", "event_count", "tag_source"]].head(30))

,tag,label,family,n_textos,n_eventos
0,ACAO_TEMERARIA,Ação temerária,Faltas e infrações de jogo,1112,6260
1,RECLAMAR_ARBITRAGEM,Reclamação à arbitragem,Comportamento e disciplina,900,2476
2,FALTA_TATICA,Impedir ataque promissor,Faltas e infrações de jogo,477,2019
3,CONDUTA_ANTIDESPORTIVA,Conduta antidesportiva,Comportamento e disciplina,835,1479
4,RETARDAR_REINICIO,Retardar o jogo,Infrações procedimentais,430,1046
5,CONFRONTO_TUMULTO,Confronto ou tumulto,Comportamento e disciplina,522,930
6,USO_IRREGULAR_BRACO,Uso irregular do braço,Faltas e infrações de jogo,187,357
7,LINGUAGEM_OFENSIVA,Ofensa verbal ou gestual,Comportamento e disciplina,244,251
8,INFRACAO_PERSISTENTE,Infração persistente,Faltas e infrações de jogo,116,229
9,COMEMORACAO_IRREGULAR,Comemoração irregular,Infrações procedimentais,140,227


,reason,event_count,tag_source


## **5. Coocorrência entre categorias**

Quantos textos carregam cada par de categorias ao mesmo tempo — evidência direta de que o problema é
genuinamente multirrótulo (uma parcela dos textos descreve mais de uma conduta na mesma justificativa).

In [6]:
tag_matrix = texts[SEMANTIC_TAGS].astype(int).to_numpy()
cooccurrence = tag_matrix.T @ tag_matrix
np.fill_diagonal(cooccurrence, 0)
cooc_df = pd.DataFrame(cooccurrence, index=SEMANTIC_TAGS, columns=SEMANTIC_TAGS)
cooc_df.to_csv(OUTPUT_DIR / "tag_cooccurrence.csv")

top_pairs = (
    cooc_df.where(np.triu(np.ones(cooc_df.shape, dtype=bool), k=1))
    .stack()
    .rename("textos_em_comum")
    .sort_values(ascending=False)
)
top_pairs = top_pairs.loc[top_pairs > 0].head(15).rename_axis(["categoria_a", "categoria_b"]).reset_index()
display(top_pairs)

,categoria_a,categoria_b,textos_em_comum
0,CONFRONTO_TUMULTO,CONDUTA_ANTIDESPORTIVA,282.0
1,ACAO_TEMERARIA,USO_IRREGULAR_BRACO,157.0
2,ACAO_TEMERARIA,JOGO_BRUSCO_GRAVE,108.0
3,RECLAMAR_ARBITRAGEM,LINGUAGEM_OFENSIVA,105.0
4,ACAO_TEMERARIA,FALTA_TATICA,89.0
5,RECLAMAR_ARBITRAGEM,AREA_TECNICA,76.0
6,ACAO_TEMERARIA,CONDUTA_ANTIDESPORTIVA,65.0
7,CONDUTA_ANTIDESPORTIVA,RETARDAR_REINICIO,63.0
8,RECLAMAR_ARBITRAGEM,CONDUTA_ANTIDESPORTIVA,57.0
9,ACAO_TEMERARIA,CONDUTA_VIOLENTA,50.0


## **6. Propagação para eventos e perfis por time, árbitro, temporada e jogador**

Os rótulos por `reason_key` retornam a todos os eventos. Como o esquema é multirrótulo, os perfis por
entidade não somam 100%: `share_within_entity` é a fração dos cartões daquela entidade que contém aquela
categoria (categorias podem coexistir no mesmo cartão).

In [7]:
tag_columns = SEMANTIC_TAGS


def build_profile(frame, dimension):
    data = frame.loc[frame[dimension].notna()].copy()
    data[dimension] = data[dimension].astype(str).str.strip()
    data = data.loc[data[dimension].ne("")]
    entity_totals = data.groupby(dimension).size().rename("entity_events")
    rows = []
    for tag in tag_columns:
        counts = data.loc[data[tag]].groupby(dimension).size()
        for entity, events_with_tag in counts.items():
            rows.append(
                {
                    "dimension": dimension,
                    "entity": entity,
                    "tag": tag,
                    "tag_label": TAG_LABELS[tag],
                    "events": int(events_with_tag),
                    "entity_events": int(entity_totals.loc[entity]),
                    "share_within_entity": events_with_tag / entity_totals.loc[entity],
                }
            )
    return pd.DataFrame(rows)


player_events = events_tagged.loc[events_tagged["participant_role"].eq("player")].copy()

season_profile = build_profile(events_tagged, "year")
team_profile = build_profile(events_tagged, "team")
referee_profile = build_profile(events_tagged, "referee")
player_profile = build_profile(player_events, "player")

season_profile.to_csv(OUTPUT_DIR / "tag_profile_by_season.csv", index=False)
team_profile.to_csv(OUTPUT_DIR / "tag_profile_by_team.csv", index=False)
referee_profile.to_csv(OUTPUT_DIR / "tag_profile_by_referee.csv", index=False)
player_profile.to_csv(OUTPUT_DIR / "tag_profile_by_player.csv", index=False)

manifest = {
    "data_path": str(DATA_PATH.relative_to(ROOT)),
    "taxonomy": TAXONOMY,
    "tags": tag_columns,
    "unclassified_tag": UNCLASSIFIED,
    "unique_texts": len(texts),
    "events": len(events),
    "events_tagged": len(events_tagged),
    "coverage_events": float((~events_tagged[UNCLASSIFIED]).mean()),
    "unclassified_share_events": float(events_tagged[UNCLASSIFIED].mean()),
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

display(
    pd.DataFrame(
        {
            "artifact": [
                "unique_text_tags.csv",
                "tag_frequency.csv",
                "tag_cooccurrence.csv",
                "card_events_tagged.csv",
                "tag_profile_by_season.csv",
                "tag_profile_by_team.csv",
                "tag_profile_by_referee.csv",
                "tag_profile_by_player.csv",
                "run_manifest.json",
                "unclassified_texts.csv",
            ],
            "directory": str(OUTPUT_DIR.relative_to(ROOT)),
        }
    )
)

,artifact,directory
0,unique_text_tags.csv,data/processed/keyword_tags
1,tag_frequency.csv,data/processed/keyword_tags
2,tag_cooccurrence.csv,data/processed/keyword_tags
3,card_events_tagged.csv,data/processed/keyword_tags
4,tag_profile_by_season.csv,data/processed/keyword_tags
5,tag_profile_by_team.csv,data/processed/keyword_tags
6,tag_profile_by_referee.csv,data/processed/keyword_tags
7,tag_profile_by_player.csv,data/processed/keyword_tags
8,run_manifest.json,data/processed/keyword_tags
9,unclassified_texts.csv,data/processed/keyword_tags
